# Deep Hedging of a European Call Option in the Black–Scholes Model

In this notebook I study how well neural networks can learn to hedge a European call option, and compare the learned strategies with the classical Black–Scholes delta hedge.

The setup is a Black–Scholes market with zero interest rate, in which the stock price follows
$$
dS_t = \sigma S_t \, dW_t, \qquad S_0 = s_0,
$$
observed at $N = 30$ equally spaced times over one month. A hedging strategy $H = (H_{t_0}, \dots, H_{t_{N-1}})$ is evaluated through the hedging loss
$$
g(S_T) - p(S_0) - \sum_{j=0}^{N-1} H_{t_j} \big( S_{t_{j+1}} - S_{t_j} \big),
$$
where $g(s) = (s - K)^+$ is the call payoff and $p$ is the Black–Scholes price.

The notebook is organised as follows:
1. **Simulation and a first deep hedging model**: one network per time step.
2. **The analytical Black–Scholes hedge**: derivation, evaluation and comparison with the learned strategies.
3. **A single time-dependent network**: one network taking both price and time as input, and a comparison of all three approaches.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
import tqdm
from scipy.stats import norm
import time
import pandas as pd
from typing import List

In [ ]:
comparison_df = pd.DataFrame(
    columns=[
        "model types",
        "mean loss",
        "mean std",
        "runtime (s)",
        "# params",
    ]
)

comparison_df["model types"] = ["many models", "single model", "analytical"]
comparison_df.set_index("model types", inplace=True)

# 1. Simulation and a first deep hedging model

### 1.1 The Black–Scholes solution

First I check that the explicit expression for $S_t$ used in the simulation indeed solves the SDE.

We verify that $S_t = s_0 \exp\left( \sigma W_t - \tfrac{1}{2} \sigma^2 t \right)$ is a solution to the following SDE:

$$
dS_t = \sigma S_t \, dW_t
$$

Ito's formula for a function $f (t, W_t)$ is:

$$
dS_t = \left( \frac{\partial S_t}{\partial t} + \tfrac{1}{2} \frac{\partial^2 S_t}{\partial W_t^2} \right) dt + \frac{\partial S_t}{\partial W_t} \, dW_t
$$

The partial derivatives of $S_t$ are given as:

$$
\frac{\partial S_t}{\partial t} = s_0 e^{\sigma W_t - \tfrac{1}{2} \sigma^2 t} \left( -\tfrac{1}{2} \sigma^2 \right) = -\tfrac{1}{2} \sigma^2 S_t
$$

$$
\frac{\partial S_t}{\partial W_t} = s_0 e^{\sigma W_t - \tfrac{1}{2} \sigma^2 t} \sigma = \sigma S_t
$$

$$
\frac{\partial^2 S_t}{\partial W_t^2} = \sigma \frac{\partial S_t}{\partial W_t} = \sigma (\sigma S_t) = \sigma^2 S_t
$$

Substituting the computed derivatives:

$$
dS_t = \left( -\tfrac{1}{2} \sigma^2 S_t + \tfrac{1}{2} \sigma^2 S_t \right) dt + \sigma S_t \, dW_t = \sigma S_t d W_t
$$

which is the Black–Scholes SDE with $r = 0$.

Alternatively, one can derive the solution directly by applying Itô's formula to the logarithm:

For $ f\left(S_t\right) = \log\left(S_t\right) $ we have: $ df = f^{'}\left(S_t\right)dS_t + \frac{1}{2}f^{''}\left(S_t\right)\left(dS_t\right)^{2} = \frac{1}{S_t}dS_t + \frac{1}{2}\left(S_t^{-2}\right)\left(S_t^{2}\sigma^{2}dt\right) = \frac{1}{S_t}\left(rS_tdt + \sigma S_tdW_t\right) - \frac{1}{2}\sigma^{2}dt = \sigma dW_t + \left(r - \frac{\sigma^{2}}{2}\right)dt $ so $ \log\left(S_t\right) =  \log\left(S_0\right) + \sigma W_t + \left(r - \frac{\sigma^{2}}{2}\right)t $ which implies that: $ S_t = s_0 \exp \left( \sigma W_t - \frac{1}{2} \sigma^2 t \right) $

### 1.2 Generating the dataset

In [ ]:
# set random seeds
np.random.seed(1)
torch.manual_seed(1)

In [ ]:
TRAIN_SIZE = int(1e6)
TEST_SIZE = int(1e5)
N = 30  # number of time steps
S_0 = 1  # initial stock price
T = 30 / 365  # one month
SIGMA = 0.5
K = 1  # strike price also known as the exercise price
R = 0  # risk-free interest rate

In [ ]:
class OptionsDataSet(Dataset):
    """Options dataset."""

    def __init__(self, size):
        self.size = size
        # Generate the training set
        z = np.random.normal(size=(size, N))
        self.data = np.zeros((size, N + 1))
        self.data[:, 0] = S_0
        for i in range(1, N + 1):
            self.data[:, i] = self.data[:, i - 1] * np.exp(
                -(SIGMA**2) / 2 * T / N + SIGMA * np.sqrt(T / N) * z[:, i - 1]
            )

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
# Generate the training set
train_dataset = OptionsDataSet(TRAIN_SIZE)

# Generate the test set
test_dataset = OptionsDataSet(TEST_SIZE)

In [ ]:
# plot 10 ranodm paths from training set
plt.figure()
for i in np.random.choice(TRAIN_SIZE, 10):
    plt.plot(train_dataset[i, :], alpha=0.5)

plt.title("10 random paths from training set")

### 1.3 Model: one network per time step

In [ ]:
# Deep hedging model: one network per time step


class HedgingModel(torch.nn.Module):
    def __init__(self):
        super(HedgingModel, self).__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(1, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1)
        )
        self.double()

    def forward(self, x):
        return self.model(x)
        self.double()

    def forward(self, x):
        return self.model(x)


models: List[torch.nn.Module] = []
for i in range(N):
    model = HedgingModel()
    models.append(model)


### 1.4 Training

In [ ]:
# Payoff function of a call option
def g(s_T):
    """
    Payoff function of a call option at final time T (=> European options)

    :param s_t: stock price at time T
    """
    return np.maximum(s_T - K, np.zeros_like(s_T))


def d_1(S_t, t):
    """
    Auxiliary term d_+ in the Black-Scholes formula.

    :param S_t: stock price at time t
    :param t: time
    """
    factor = 1 / (SIGMA * np.sqrt(T - t))
    return factor * (np.log(S_t / K) + (0 + SIGMA**2 / 2) * (T - t))


def d_2(S_t, t):
    """
    Auxiliary term d_- in the Black-Scholes formula.

    :param S_t: stock price at time t
    :param t: time
    """
    return d_1(S_t, t) - SIGMA * np.sqrt(T - t)


# Black-Scholes price of a call option with given strike and maturity


def C(S_t, t):
    """
    Value of European call option at time t given stock price S_t

    :param S_t: stock price at time t
    :param t: time
    """
    return norm.cdf(d_1(S_t, t)) * S_t - norm.cdf(d_2(S_t, t)) * K * np.exp(R * (T - t))


def p(s_t):
    """
    Risk neutral price of a call option

    :param s_t: current stock price
    """
    return C(s_t, 0)


def loss_function(H: torch.Tensor, S: torch.Tensor):
    """

    :param H: model output of shape (batch_size, N)
    :param S: stock prices of shape (batch_size, N + 1)
    """

    return torch.mean(
        (g(S[:, -1]) - p(S[:, 0].numpy()) - torch.sum(H * (S[:, 1:] - S[:, :-1]), dim=1)) ** 2
    )

In [ ]:
### Training preperation

# Define training hyperparameters
n_epochs = 5
lr = 1e-2
batch_size = 1000
milestones = [75]

# Instantiate an optimizer (i.e. a (Stochastic) Gradient Descent algorithm)
all_params = []
for model in models:
    all_params += list(model.parameters())

optimizer = torch.optim.Adam(all_params, lr=lr)

# Instantiate a scheduler to decrease the learning rate during training according to a pre-defined rule
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=milestones, gamma=0.1
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

In [ ]:
for model in models:
    model.train()
# Train the model
losses = []
for epoch in range(n_epochs):
    # At each epoch iterate over batches
    training_loop = tqdm.tqdm(train_loader, total=len(train_loader))
    training_loop.set_description(f"Epoch {epoch + 1}/{n_epochs}")
    for X_batch in training_loop:
        H = []
        for i, model in enumerate(models):
            H.append(model(torch.log(X_batch[:, i].unsqueeze(1)).double()))

        H = torch.stack(H, dim=1).squeeze(2)
        # compute training loss on batch
        loss = loss_function(H, X_batch)
        # compute gradient of training loss wrt parameters
        loss.backward()
        # do one gradient step
        optimizer.step()
        # reset the computational graph
        optimizer.zero_grad()

        # Print loss in the progression bar
        if training_loop.n % 10 == 0:
            training_loop.set_postfix(loss=loss.item())

    # decrease learning rate according to scheduler's settings
    scheduler.step()

    # save training loss for plotting
    losses.append(loss.item())

plt.plot(losses, label="Training loss")
plt.show()

### 1.5 Evaluation

In [ ]:
def loss_function_eval(H: torch.Tensor, S: torch.Tensor):
    """

    :param H: model output of shape (batch_size, N)
    :param S: stock prices of shape (batch_size, N + 1)
    """

    return g(S[:, -1]) - p(S[:, 0].numpy()) - torch.sum(H * (S[:, 1:] - S[:, :-1]), dim=1)

In [ ]:
def evaluate_models(models, test_loader):
    """
    Evaluate the model on the test set

    :param test_loader: test data loader
    """

    losses = []
    with torch.no_grad():
        for X_batch in test_loader:
            H = []

            for i, model in enumerate(models):
                H.append(model(torch.log(X_batch[:, i].unsqueeze(1)).double()))

            H = torch.stack(H, dim=1).squeeze(2)
            loss = loss_function_eval(H, X_batch)
            losses.append(loss)

    return torch.cat(losses).detach().numpy()

In [ ]:
# Evaluate the hedging losses on the test set

# Compute the hedging losses on the test set
hedging_losses = []


start = time.time()

hedging_losses = evaluate_models(models, test_loader)

# Plot a histogram of the hedging losses and print mean and standard deviation
plt.figure()
plt.hist(hedging_losses, bins=100)
plt.title("Hedging losses (multiple models)")
plt.xlabel("Hedging loss")
plt.ylabel("Frequency")

end = time.time()
many_run_time = end - start
many_mean = np.mean(hedging_losses)
many_std = np.std(hedging_losses)

comparison_df.loc["many models", "mean loss"] = many_mean
comparison_df.loc["many models", "mean std"] = many_std
comparison_df.loc["many models", "runtime (s)"] = many_run_time
comparison_df.loc["many models", "# params"] = sum(p.numel() for p in all_params)

print("mean", many_mean)
print("standard deviation", np.std(hedging_losses))

To check whether the networks are trained well, I look at whether the losses are centred around zero and whether their standard deviation is small.

The losses are centred around zero and their standard deviation is small, so the models are trained well.

# 2. The analytical Black–Scholes hedge

### 2.1 Derivation of the delta hedge

In the Black–Scholes model the hedging strategy is the delta of the option,
$$
H^{\text{BS}}_t(s) = \frac{\partial C(s, t)}{\partial s}.
$$
I derive it step by step.

$$ H^{\text{BS}}_t(s) =  \frac{\partial C(s, t)}{\partial s} = \frac{\partial (\Phi(d_{+}(s))s-\Phi(d_{-}(s))Ke^{-r(T-t)})}{\partial s} $$
$$= \Phi\left(d_+\left(s\right)\right) + s\Phi^{'}\left(d_+\left(s\right)\right)d_+^{'}\left(s\right)  - Ke^{-r\left(T-t\right)}\Phi^{'}\left(d_-\left(s\right)\right)d_-^{'}\left(s\right) $$
Let's call $\Phi'$, $f$. Since $\Phi$ is the standard normal cdf, we know $f$ is the standard normal pdf. Using the fact that $d_+^{'} = d_-^{'}$ (because their difference doesn't depend on $s$)
$$= \Phi\left(d_+\left(s\right)\right) + sf\left(d_+\left(s\right)\right)d_+^{'}\left(s\right)  - Ke^{-r\left(T-t\right)}f\left(d_-\left(s\right)\right)d_+^{'}\left(s\right) $$

Calculating $d'_{+}(s)$ we get $ \frac{1}{\sigma \sqrt{T-t}} \frac{1}{s} = \frac{1}{s\sigma \sqrt{T-t}}$, so we get

$$ H^{\text{BS}}_t(s) = \Phi\left(d_+\left(s\right)\right) + sf\left(d_+\left(s\right)\right)\frac{1}{s\sigma \sqrt{T-t}}  - Ke^{-r\left(T-t\right)}f\left(d_-\left(s\right)\right)\frac{1}{s\sigma \sqrt{T-t}} $$
$$= \Phi\left(d_+\left(s\right)\right) + f\left(d_+\left(s\right)\right)\frac{1}{\sigma \sqrt{T-t}}  - Ke^{-r\left(T-t\right)}f\left(d_-\left(s\right)\right)\frac{1}{s\sigma \sqrt{T-t}}$$

where $f$ is the standard gaussian pdf

### 2.2 Evaluation of the analytical hedge

In [ ]:
# Analytical (delta) hedge derived above
def analytical_sol(s, t):
    # We can forget about the exponential term because r = 0
    return (
        norm.cdf(d_1(s, t))
        + norm.pdf(d_1(s, t)) / (SIGMA * np.sqrt(T - t))
        - K * norm.pdf(d_2(s, t)) / (s * SIGMA * np.sqrt(T - t))
    )


# Evaluate the losses of the analytical hedging strategies on the test set

In [ ]:
def evaluate_analytical_hedge(test_loader):
    """
    Evaluate the analytical hedge on the test set

    :param test_loader: test data loader
    """

    losses = []
    all_timesteps = np.arange(0, N) / N * T
    for X_batch in tqdm.tqdm(test_loader):
        X = X_batch.detach().numpy()

        H = analytical_sol(X[:, :-1], all_timesteps)

        loss = g(X[:, -1]) - p(X[:, 0]) - np.sum(H * (X[:, 1:] - X[:, :-1]), axis=-1)

        losses.append(loss)

    return np.concatenate(losses)




In [ ]:

start = time.time()

analytical_losses = evaluate_analytical_hedge(test_loader)
# Plot a histogram of the hedging losses and print mean and standard deviation
plt.figure()
plt.hist(hedging_losses, bins=100)
plt.title("Hedging losses (analytical)")
plt.xlabel("Hedging loss")
plt.ylabel("Frequency")

end = time.time()
analytical_time = end - start
analytical_mean = np.mean(analytical_losses)
analytical_std = np.std(analytical_losses)

comparison_df.loc["analytical", "mean loss"] = analytical_mean
comparison_df.loc["analytical", "mean std"] = analytical_std
comparison_df.loc["analytical", "runtime (s)"] = analytical_time
comparison_df.loc["analytical", "# params"] = 0

print("mean", analytical_mean)
print("standard deviation", analytical_std)

In the Black–Scholes model every option can be perfectly hedged by its analytical hedging strategy. Nevertheless, the analytical hedge above does not give zero losses.

The reason is that the analytical strategy is the perfect hedge in continuous time, while here we can only rebalance at discrete times.

### 2.3 Comparison of hedging losses

Next I compare the deep hedging model with the analytical strategy.

The losses of our hedging models have approximately the same standard deviation as the losses of the analytical model, so their performance is approximately the same.

### 2.4 Comparison of the learned and analytical strategies

In [ ]:
# Plot both the neural network strategies and the analytical hedging strategies

fig, axs = plt.subplots(2, 2, figsize=(12, 6))

for i, t_j in enumerate([0, 10, 20, 29]):
    ax = axs.flatten()[i]

    # plot s |-> H(s, t_j) for multi-model and analytical hedging strategies for s in [0.5, 1.5]
    s = np.linspace(0.5, 1.5, 100)
    model = models[t_j]
    H_model = model(torch.log(torch.tensor(s).unsqueeze(1)).double()).detach().numpy()
    ax.plot(s, H_model, label="Multi-model")

    H_analytical = analytical_sol(s, t_j / N * T)
    ax.plot(s, H_analytical, label="Analytical")

    ax.set_title(f"Hedging strategy at time $t_j={t_j}$")
    ax.set_xlabel("Stock price")
    ax.set_ylabel("Hedge")

    ax.legend()

plt.tight_layout()
plt.show()


The two functions are most similar towards the end of the time period, when the option is about to expire.

This is because, with very little time until expiry, the stock price can only change slightly before maturity. With so little uncertainty left, any reasonable strategy behaves very similarly.

# 3. A single time-dependent network

### 3.1 Model

In [ ]:
# Single deep hedging model taking (log-price, time) as input
class HedgingModelTime(torch.nn.Module):
    def __init__(self):
        super(HedgingModelTime, self).__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(2, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
        )
        self.double()

    def forward(self, x):
        return self.model(x)



### 3.2 Training

In [ ]:
# Train the time-dependent model

# n_epochs = 5 # number of epochs uncomment if different from other training
# lr = 1e-2
# milestones = [75]

time_model = HedgingModelTime()
losses = []

optimizer = torch.optim.Adam(time_model.parameters(), lr=lr)

# Instantiate a scheduler to decrease the learning rate during training according to a pre-defined rule
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=milestones, gamma=0.1
)

# Train the model
losses = []
for epoch in range(n_epochs):
    # At each epoch iterate over batches
    training_loop = tqdm.tqdm(train_loader, total=len(train_loader))
    training_loop.set_description(f"Epoch {epoch + 1}/{n_epochs}")
    for X_batch in training_loop:
        M = X_batch.shape[0]

        t = torch.arange(0, N) / N * T
        time_input = torch.sqrt(T - t)
        time_input = time_input.repeat(M, 1)
        s_input = torch.log(X_batch[:, :N])
        #full_input = torch.stack((s_input, time_input)).reshape(-1, 2)
        full_input = torch.stack((s_input, time_input),dim=2)
        model_output = time_model(full_input)
        H = model_output.squeeze(-1)

        # compute training loss on batch
        loss = loss_function(H, X_batch)
        # compute gradient of training loss wrt parameters
        loss.backward()
        # do one gradient step
        optimizer.step()
        # reset the computational graph
        optimizer.zero_grad()

        # Print loss in the progression bar
        if training_loop.n % 10 == 0:
            training_loop.set_postfix(loss=loss.item())

    # decrease learning rate according to scheduler's settings
    scheduler.step()

    # save training loss for plotting
    losses.append(loss.item())

plt.plot(losses, label="Training loss time model")

### 3.3 Evaluation

In [ ]:
def evaluate_time_model(model, test_loader):
    """
    Evaluate the model on the test set

    :param test_loader: test data loader
    """

    losses = []
    with torch.no_grad():
        for X_batch in test_loader:
            M = X_batch.shape[0]
            t = torch.arange(0, N) / N * T
            time_input = torch.sqrt(T - t)
            time_input = time_input.repeat(M, 1)
            s_input = torch.log(X_batch[:, :N])
            #full_input = torch.stack((s_input, time_input)).reshape(-1, 2)
            full_input = torch.stack((s_input, time_input),dim=2)
            model_output = time_model(full_input)
            H = model_output.squeeze(-1)
            loss = loss_function_eval(H, X_batch)
            losses.append(loss)

    return torch.cat(losses).detach().numpy()

In [ ]:
# Evaluate the hedging losses on the test set

# Compute the hedging losses on the test set


hedging_losses = []

start = time.time()

hedging_losses = evaluate_time_model(time_model, test_loader)

# Plot a histogram of the hedging losses and print mean and standard deviation
plt.figure()
plt.hist(hedging_losses, bins=100)
plt.title("Hedging losses (single model)")
plt.xlabel("Hedging loss")
plt.ylabel("Frequency")

end = time.time()
time_run_time = end - start
time_mean = np.mean(hedging_losses)
time_std = np.std(hedging_losses)

comparison_df.loc["single model", "mean loss"] = time_mean
comparison_df.loc["single model", "mean std"] = time_std
comparison_df.loc["single model", "runtime (s)"] = time_run_time
comparison_df.loc["single model", "# params"] = sum(
    p.numel() for p in time_model.parameters()
)


print("mean", np.mean(hedging_losses))
print("standard deviation", np.std(hedging_losses))

### 3.4 Comparison of the three approaches

In [ ]:
# Comparison of run-time, performance and number of parameters
comparison_df

The analytical solution is both the fastest and the best performing (though all performances are pretty much equal). The single model is slowest, because we added a lot of layer's trying to get it's performance to same level as the others.

The many-models model performs very well and fast, but has the disadvantage of needing a different model for every discrete time step, meaning if we have a large number of discrete time steps (either because the options take a long time to expire or because we are being very granular with time) we will have to optimize a very large number of parameters.

Because of this, plus the fact that the analytical model will perform better if we the more granular we are with time (because we are closer to the continous time case), I would choose the analytical model in cases where there are a large number of discrete time steps and the many-models model otherwise.


